# Advanced RAG

## Goal

Advanced RAG improves the **evidence** before the LLM answers.

Pipeline:

`Chunk → Semantic Search + Keyword Search → Hybrid Search → Rerank → Compress Context → Evaluate`

In [ ]:
%pip install -q sentence-transformers numpy pandas

In [ ]:
docs = [
    {"source":"rag.txt","text":"RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from an external knowledge base and gives it to a language model as context."},
    {"source":"python.txt","text":"Python is a programming language used for web development, data science, machine learning, automation and AI."},
    {"source":"ml.txt","text":"Machine learning is a branch of AI. Supervised learning uses labeled data. Unsupervised learning finds patterns without labels. Reinforcement learning learns through rewards."},
    {"source":"faiss.txt","text":"FAISS is a library for efficient similarity search and clustering of dense vectors. It supports L2 and inner-product search."},
]

def chunks(text, size=120, overlap=20):
    out=[]; start=0
    while start < len(text):
        p=text[start:start+size].strip()
        if p: out.append(p)
        start += size-overlap
    return out

items=[]
for d in docs:
    for i,p in enumerate(chunks(d["text"])):
        items.append({"source":d["source"],"chunk_id":i,"text":p})

from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np, re

embedder = SentenceTransformer("all-MiniLM-L6-v2")
vectors = embedder.encode([x["text"] for x in items], normalize_embeddings=True)

## 1. Semantic search

Search by meaning.

In [ ]:
def semantic_search(query, k=5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = vectors @ q
    order = np.argsort(scores)[::-1][:k]
    return [{**items[i], "semantic_score":float(scores[i])} for i in order]

for x in semantic_search("How does RAG work?"):
    print(round(x["semantic_score"],3), x["source"], x["text"])

## 2. Keyword search

Exact words are useful for IDs, error codes, names and technical terms.

In [ ]:
STOP={"what","is","the","a","an","of","to","in","and","for","on","with","how","does"}

def keyword_search(query, k=5):
    words=[w for w in re.findall(r"\b\w+\b",query.lower()) if w not in STOP]
    scored=[]
    for x in items:
        text_words=set(re.findall(r"\b\w+\b",x["text"].lower()))
        score=sum(w in text_words for w in words)
        scored.append({**x,"keyword_score":score})
    return sorted(scored,key=lambda x:x["keyword_score"],reverse=True)[:k]

for x in keyword_search("What is RAG?"):
    print(x["keyword_score"], x["source"])

## 3. Hybrid search

Combine semantic and keyword signals.

The weight should be tested on your own data.

In [ ]:
def norm(a):
    a=np.array(a,dtype=float)
    return np.ones_like(a) if a.max()==a.min() else (a-a.min())/(a.max()-a.min())

def hybrid_search(query, k=5, semantic_weight=0.7):
    q=embedder.encode([query],normalize_embeddings=True)[0]
    semantic=vectors@q

    words=[w for w in re.findall(r"\b\w+\b",query.lower()) if w not in STOP]
    keyword=[]
    for x in items:
        tw=set(re.findall(r"\b\w+\b",x["text"].lower()))
        keyword.append(sum(w in tw for w in words))

    score=semantic_weight*norm(semantic)+(1-semantic_weight)*norm(keyword)
    order=np.argsort(score)[::-1][:k]

    return [{**items[i],"hybrid_score":float(score[i])} for i in order]

for x in hybrid_search("What is RAG?"):
    print(round(x["hybrid_score"],3), x["source"], x["text"])

## 4. Reranking

Pattern:

`retrieve many → rerank → keep a few`

A cross-encoder judges the query and candidate together. The supplied Chroma notebook uses this idea.

In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

question="What is RAG?"
candidates=hybrid_search(question,k=min(8,len(items)))
scores=reranker.predict([[question,x["text"]] for x in candidates])

ranked=[{**x,"rerank_score":float(s)} for x,s in zip(candidates,scores)]
ranked.sort(key=lambda x:x["rerank_score"],reverse=True)

for x in ranked:
    print(round(x["rerank_score"],3),x["source"],x["text"])

## 5. Context compression

Keep only the best few chunks.

More text is not automatically better.

In [ ]:
top=ranked[:2]
context="\n\n".join(
    f"[Source: {x['source']}, Chunk: {x['chunk_id']}]\n{x['text']}"
    for x in top
)
print(context)

## 6. Evaluation

Useful RAG metrics:
- **Context precision:** relevant chunks ranked above irrelevant chunks
- **Context recall:** necessary information was retrieved
- **Faithfulness:** answer claims are supported by retrieved context
- **Answer relevance:** answer addresses the question

The supplied notebook has a simpler source-hit evaluation. Start simple, then make it stronger.

In [ ]:
tests=[
    ("What does RAG stand for?","rag.txt"),
    ("What is Python used for?","python.txt"),
    ("What are the types of machine learning?","ml.txt"),
]

def source_recall(weight=0.7,k=3):
    hits=0
    for q,expected in tests:
        result=hybrid_search(q,k=k,semantic_weight=weight)
        sources={x["source"] for x in result}
        ok=expected in sources
        hits += int(ok)
        print("PASS" if ok else "FAIL", q, sources)
    return hits/len(tests)

print("recall:",source_recall())

## 7. Improvement loop

Experiment with one variable at a time:
- chunk size
- overlap
- hybrid weight
- candidate count
- rerank top-k
- final context size

Track quality, latency and cost.

`Measure → change one thing → measure again`

In [ ]:
for weight in [0.5,0.7,0.9]:
    print("weight",weight,"recall",source_recall(weight=weight,k=3))